<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study/12_Cassava_Leaf_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

Mounted at /content/drive


In [1]:
#!/bin/bash
!kaggle datasets download nirmalsankalana/cassava-leaf-disease-classification

Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/cassava-leaf-disease-classification
License(s): CC0-1.0
100% 2.39G/2.39G [00:31<00:00, 81.7MB/s]



In [2]:
!unzip -q cassava-leaf-disease-classification.zip -d ./data

In [3]:
!ls -l ./data/data/

total 772
drwxr-xr-x 2 root root  40960 Jun  9 11:36 Cassava___bacterial_blight
drwxr-xr-x 2 root root  69632 Jun  9 11:37 Cassava___brown_streak_disease
drwxr-xr-x 2 root root  73728 Jun  9 11:37 Cassava___green_mottle
drwxr-xr-x 2 root root  90112 Jun  9 11:37 Cassava___healthy
drwxr-xr-x 2 root root 499712 Jun  9 11:37 Cassava___mosaic_disease


In [14]:
import os
import torch
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
import numpy as np

BATCH_SIZE = 32

# 데이터 경로 설정
data_dir = './data/data'

# 기본 전처리 정의
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), # 회전 조작 (질병 얼룩 각도 비틀기)
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기, 대비 조작
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation 데이터셋은 모델 평가용이므로 순수한 본래 이미지 상태를 유지해야 함
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset_train = ImageFolder(root=data_dir, transform=transform_train)
full_dataset_val = ImageFolder(root=data_dir, transform=transform_val)

# 시드 고정 및 데이터 분할
targets = np.array(full_dataset_train.targets)
train_size = int(0.8 * len(full_dataset_train))
val_size = len(full_dataset_train) - train_size

# 인덱스를 기준으로 똑같이 나누되 전처리만 다르게 적용하기 위함
indices = np.arange(len(full_dataset_train))
np.random.seed(42)
np.random.shuffle(indices)
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_dataset = torch.utils.data.Subset(full_dataset_train, train_idx)
val_dataset = torch.utils.data.Subset(full_dataset_val, val_idx)

# 불균형 데이터셋 가중치 재계산
train_targets = targets[train_idx]
class_counts =np.bincount(train_targets)
clsss_weights = 1.0 / class_counts
sample_weights = clsss_weights[train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)



# ImageFolder로 데이터셋 자동 로드
# full_dataset = ImageFolder(root=data_dir, transform=transform)

# 클래스 이름과 매핑된 인덱스 확인
# print("클래스 매핑:", full_dataset.class_to_idx)

# Train / Validation 분리 (8:2)
# train_size = int(0.8 * len(full_dataset))
# val_size = len(full_dataset) - train_size
# train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# [핵심] 불균형 해결을 위한 클래스별 가중치 계산
# 전체 데이터셋의 라벨 배열 추출
# targets = np.array(full_dataset.targets)
# train_idx = train_dataset.indices
# train_targets = targets[train_idx]

# 각 클래스별 데이터 개수 세기
# class_counts = np.bincount(train_targets)
# class_weights = 1.0 / class_counts

# 학습 데이터셋의 각 샘플에 해당하는 가중치 리스트 생성
# sample_weights = class_weights[train_targets]

# Sampler 정의
# sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoader 구축 (셔플 대신 sampler를 주입)
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [15]:
images, labels = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


In [16]:
import timm
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# 5개의 클래스를 분류하는 프리트레인 모델 로드
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# T_max는 전체 에폭 수를 의미합니다. 여기서는 3에폭 동안 완만하게 줄이도록 세팅합니다.
# eta_min은 학습률이 떨어질 최저 마지노선입니다.
scheduler = CosineAnnealingLR(optimizer, T_max=3, eta_min=1e-6)

In [17]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # ce_loss: 일단 일반적인 틀린 점수(Cross Entropy)를 구합니다.
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        # exp(-ce_loss): 모델이 이 문제를 맞힐 확률(0.0~1.0)을 뜻합니다.
        pt = torch.exp(-ce_loss)
        # gamma = 2라고 가정
        # 다수 클래스(쉬운 문제): 너무 흔해서 이미 잘 맞힙니다.
        # pt = 0.9라면 0.1 ** 2 = 0.01. 벌점을 원래의 1% 수준으로 대폭 삭감합니다.

        # 소수 클래스(어려원 문제): 데이터가 없어서 잘 못 맞힙니다.
        # pt = 0.1이라면 0.9 ** 2 = 0.81. 벌점을 거의 그대로(81%) 유지합니다.
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=1, gamma=2).to(device)

In [19]:
from sklearn.metrics import f1_score
import numpy as np

epochs = 3

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_train_loss = train_loss / len(train_dataset)
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_macro_f1 = f1_score(all_labels, all_preds, average='macro')

    # 현재 에폭에 설정된 학습률 정보 가져오기
    current_lr = scheduler.get_last_lr()[0]

    # 한 에폭의 배치가 끝났으니 다음 에폭을 위해 학습률을 깎아줍니다.
    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | "\
          f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Macro F1: {epoch_macro_f1:.4f}")


Epoch [1/3] | LR: 0.000100 | Train Loss: 0.5933 | Val Loss: 0.6325 | Val Macro F1: 0.5591
Epoch [2/3] | LR: 0.000075 | Train Loss: 0.3977 | Val Loss: 0.5221 | Val Macro F1: 0.5963
Epoch [3/3] | LR: 0.000026 | Train Loss: 0.3167 | Val Loss: 0.4751 | Val Macro F1: 0.6073
